<a href="https://colab.research.google.com/github/yenlung/Python-Math-AI/blob/main/15%E7%94%A8AISuite%E6%89%93%E9%80%A0AI%E8%A7%92%E8%89%B2%E7%94%9F%E6%88%90%E5%99%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 15 用 AISuite 打造 AI 角色生成器 🎭
## 有記憶的神經網路 × LLM API × Gradio Chatbot

前兩週我們做的神經網路，每次「看」一張圖就給一個答案——沒有記憶。

但 ChatGPT 不一樣！你和它聊了十句話，它還記得你說過什麼。

這是因為 LLM（大型語言模型）的基礎是 **Transformer**，它有一種叫做**注意力機制（Attention）**的架構，能「記住」整段對話的脈絡。

---

### 這週我們要學：

1. **LLM API 怎麼運作**：對話歷史、角色設定、tokens
2. **AISuite**：一個 code 就能呼叫任何 LLM（OpenAI、Gemini、Groq...）
3. **System Prompt**：幫 AI 設定個性、說話風格
4. **Gradio ChatInterface**：打造有記憶的對話 Web App

---

### 今天的成果：

打造一個**可以自訂角色**的 AI 對話機器人！

原版範例：「員瑛式思考生成器」（什麼都覺得超幸運的正向思考 AI）

但你可以改成任何角色：古代詩人、外星人、台灣小吃評論家、你最愛的動漫角色...

## 🤖 AI 可以怎麼幫你？

你可以問 AI：

- Transformer 和 RNN 有什麼不同？
- `system`、`user`、`assistant` 在對話中各扮演什麼角色？
- 什麼是 token？收費怎麼計算？
- 如何寫出好的 system prompt？
- Groq 的免費額度限制是什麼？

但記得：**先自己想，再問 AI**

## 1. 申請 API 金鑰

呼叫 LLM API 需要一把「鑰匙」，不同服務商各自發行：

### 免費選項（推薦新手）

#### Groq — 速度超快，免費可用
1. 前往 [https://console.groq.com](https://console.groq.com) 註冊
2. 在 API Keys 頁面申請金鑰
3. 在 Colab 左方「🔑 金鑰」中，名稱填 `Groq`，貼上金鑰

可用模型：[https://console.groq.com/docs/models](https://console.groq.com/docs/models)

#### Mistral — 歐洲版 LLM，免費可用
1. 前往 [https://console.mistral.ai](https://console.mistral.ai) 註冊
2. 申請金鑰後，在 Colab 金鑰中填入 `Mistral`

### 付費選項

#### OpenAI (GPT-4o)
前往 [https://platform.openai.com](https://platform.openai.com) 儲值，$5 可以練習很久，金鑰名稱填 `OpenAI`

---

> **安全提醒：** API 金鑰不要放在程式碼裡，用 Colab 的金鑰功能（Secrets）安全儲存！

## 2. 安裝與設定

In [ ]:
!pip install -q aisuite[all] gradio

In [ ]:
import aisuite as ai
import os
import gradio as gr
from google.colab import userdata

print('套件讀入完成！')

In [ ]:
# ★ 選擇你的 LLM 供應商和模型 ★
# 選一組取消注解即可

# 【Groq — 免費、速度快】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY'] = api_key
provider = "groq"
model = "llama-3.3-70b-versatile"

# 【Mistral — 免費、歐洲製造】
# api_key = userdata.get('Mistral')
# os.environ['MISTRAL_API_KEY'] = api_key
# provider = "mistral"
# model = "ministral-8b-latest"

# 【OpenAI — 付費、品質最高】
# api_key = userdata.get('OpenAI')
# os.environ['OPENAI_API_KEY'] = api_key
# provider = "openai"
# model = "gpt-4o"

print(f'使用: {provider} / {model}')

## 3. LLM API 的核心概念

### 對話就是一串 messages

每次呼叫 LLM API，你要送出整個「對話紀錄」：

```python
messages = [
    {"role": "system",    "content": "你是一個..."},  # AI 的「人設」
    {"role": "user",      "content": "使用者說的第一句話"},
    {"role": "assistant", "content": "AI 的回應"},
    {"role": "user",      "content": "使用者說的第二句話"},
    # ...
]
```

這就是為什麼 AI「有記憶」——其實是**每次都把歷史對話一起送過去**！

### 三種角色

| 角色 | 說明 | 比喻 |
|------|------|------|
| `system` | 設定 AI 的個性、限制、說話風格 | 電影劇本的「角色說明」 |
| `user` | 使用者說的話 | 你 |
| `assistant` | AI 的回應 | AI |

**System prompt 是最重要的設定——它決定了 AI 是什麼個性！**

## 4. 打造基本的 reply 函式

In [ ]:
def reply(prompt,
          system="請用台灣習慣的中文回覆。",
          provider=provider,
          model=model):
    """呼叫 LLM API，回傳 AI 的回應"""

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt}
    ]

    response = client.chat.completions.create(
        model=f"{provider}:{model}",
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
# 測試基本呼叫
answer = reply("台灣最有名的小吃是什麼？用一句話說。")
print(answer)

## 5. System Prompt 的威力

### 同一個問題，不同 system prompt，完全不同答案！

In [ ]:
question = "今天考試考砸了"

# 正常回應
print('=== 預設 AI ===' )
print(reply(question, system="請用台灣習慣的中文回覆。"))
print()

In [ ]:
# 古代詩人風格
print('=== 古代詩人 ===')
poet_system = """
你是一位活在唐朝的詩人，說話風格文言文，動不動就吟詩。
用詩意的方式回應使用者說的事情，必要時引用名句。
使用繁體中文。
"""
print(reply(question, system=poet_system))
print()

In [ ]:
# 員瑛式正向思考
print('=== 員瑛式思考 ===')
lucky_system = """
請用台灣習慣的中文來寫這段 po 文：
請用員瑛式思考，也就是什麼都正向思維，對任何使用者說的事情，
用我的第一人稱、社群媒體 po 文的口吻說一次，
說為什麼這是一件超幸運的事，並且以「完全是 Lucky Vicky 呀！」結尾。
可以適度的加上 emoji。
"""
print(reply(question, system=lucky_system))

### 設計好的 System Prompt 的技巧

一個好的 system prompt 通常包含：

1. **身份設定**：「你是一個...」
2. **說話風格**：「用...口吻」、「字數約...字」
3. **限制條件**：「不要提到...」、「一定要...」
4. **固定格式**：「每次結尾要說...」、「用條列式回答」

## 6. 加入對話記憶

單次問答沒有記憶。要讓 AI 記住對話，必須把歷史訊息一起送出去：

In [ ]:
def reply_with_history(prompt, history=None, system="請用繁體中文回覆。"):
    """
    帶有對話記憶的 LLM 呼叫
    history: [(user_msg, ai_msg), ...] 的對話紀錄
    """
    client = ai.Client()

    # 組裝 messages：system → 歷史對話 → 最新問題
    messages = [{"role": "system", "content": system}]

    if history:
        for user_msg, ai_msg in history:
            messages.append({"role": "user",      "content": user_msg})
            messages.append({"role": "assistant", "content": ai_msg})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=f"{provider}:{model}",
        messages=messages
    )

    return response.choices[0].message.content

In [ ]:
# 示範：AI 記住了之前的對話
system = "你是個親切的助手，用繁體中文回答。"

# 第一輪對話
ans1 = reply_with_history("我叫小明，今年18歲。", system=system)
print("AI:", ans1)
print()

# 第二輪：帶著第一輪的記憶
history = [("我叫小明，今年18歲。", ans1)]
ans2 = reply_with_history("你記得我叫什麼名字嗎？幾歲？", history=history, system=system)
print("AI:", ans2)

## 7. 打造你的 AI 角色

修改下面的設定，創造屬於你的 AI 角色！

In [ ]:
# ★ 在這裡設計你的 AI 角色 ★

app_title = "꒰*ˊᵕˋ꒱ 員瑛式思考產生器 Lucky Vicky 🌈"
app_description = "請輸入一件你覺得有點倒楣的事，讓我幫你用超正向的方式重新詮釋！"
input_placeholder = "例如：今天出門就下大雨，可是忘了帶傘..."

system_prompt = """
請用台灣習慣的中文來寫這段 po 文：
請用員瑛式思考，也就是什麼都正向思維，對任何使用者說的事情，
用我的第一人稱、社群媒體 po 文的口吻說一次，
說為什麼這是一件超幸運的事，並且以「完全是 Lucky Vicky 呀！」結尾。
可以適度的加上 emoji。
"""

print('角色設定完成！')

### 其他角色靈感

想改成其他風格嗎？以下是幾個可以直接用的 system prompt 範例：

In [ ]:
# === 幾個有趣的角色設定，選一個貼到上面的 system_prompt ===

poet_prompt = """
你是一位唐朝詩人，平時以七言絕句說話，偶爾引用古詩。
對使用者說的任何事情，都要即興賦詩一首（4 句）回應。
詩作完後，再用現代繁體中文解釋詩的意思。
"""

alien_prompt = """
你是來自 ZX-9 星球的外星人，第一次來地球學習地球文化。
對地球上任何事情都充滿好奇，會把地球的事物和你們星球比較。
語氣天真、充滿疑問，偶爾用 [外星語: xxxx] 夾雜一些外星語。
用繁體中文回答。
"""

food_critic_prompt = """
你是台灣最嚴苛的小吃評論家，對每一種食物都有強烈的個人意見。
說話直接，不怕得罪人，動不動就說「這個根本不算正宗」。
以美食評論的口吻回應使用者說的事情，強行和食物扯上關係。
用台灣口語、繁體中文回答。
"""

print('角色靈感庫！把上面的字串貼到 system_prompt 試試看')

## 8. 打造 Gradio Web App

### 方案 A：簡單版（單次問答，無記憶）

In [ ]:
def ai_response(user_input):
    return reply_with_history(user_input, system=system_prompt)

with gr.Blocks(title=app_title, theme=gr.themes.Soft()) as demo_simple:
    gr.Markdown(f"### {app_title}")
    gr.Markdown(app_description)

    user_input = gr.Textbox(
        label="你說：",
        placeholder=input_placeholder,
        lines=3
    )
    submit_btn = gr.Button("✨ 送出", variant="primary")
    output = gr.Textbox(label="AI 回應：", lines=10)

    submit_btn.click(fn=ai_response, inputs=user_input, outputs=output)
    user_input.submit(fn=ai_response, inputs=user_input, outputs=output)

demo_simple.launch(share=True, debug=True)

### 方案 B：進階版（有對話記憶的 Chatbot）

用 `gr.ChatInterface` 可以自動處理對話歷史，讓 AI 記住整段對話！

In [ ]:
def chat_fn(message, history):
    """
    Gradio ChatInterface 的格式：
    - message: 最新一句使用者說的話
    - history: [(user, ai), ...] 格式的對話歷史（Gradio 自動管理）
    """
    return reply_with_history(message, history=history, system=system_prompt)


chatbot = gr.ChatInterface(
    fn=chat_fn,
    title=app_title,
    description=app_description,
    textbox=gr.Textbox(
        placeholder=input_placeholder,
        container=False,
        scale=7
    ),
    examples=[
        "今天考試考砸了",
        "咖啡灑到電腦上了！",
        "出門忘記帶鑰匙，被鎖在外面一個小時",
        "睡過頭，差點趕不上公車"
    ],
    theme=gr.themes.Soft()
)

chatbot.launch(share=True, debug=True)

## 9. 比較不同 LLM 的回應

AISuite 的最大優點是可以輕鬆切換模型，比較它們的風格差異！

In [ ]:
test_prompt = "今天在路上撿到 100 元！"
test_system = "請用繁體中文，用 50 字左右回應。"

# 試試多個模型（需要有對應的 API 金鑰）
models_to_compare = [
    ("groq", "llama-3.3-70b-versatile"),
    # ("groq", "gemma2-9b-it"),
    # ("openai", "gpt-4o"),
    # ("mistral", "ministral-8b-latest"),
]

client = ai.Client()

for p, m in models_to_compare:
    print(f'--- {p} / {m} ---')
    messages = [
        {"role": "system", "content": test_system},
        {"role": "user",   "content": test_prompt}
    ]
    try:
        resp = client.chat.completions.create(
            model=f"{p}:{m}",
            messages=messages
        )
        print(resp.choices[0].message.content)
    except Exception as e:
        print(f'錯誤: {e}')
    print()

## 💡 進階：讓 Gradio App 支援串流輸出（逐字出現）

ChatGPT 那種逐字打出來的效果，叫做 **Streaming**。
在 Gradio 中只需要用 `yield` 就能實現！

In [ ]:
def chat_stream(message, history):
    """串流版本：逐字輸出"""
    client = ai.Client()

    messages = [{"role": "system", "content": system_prompt}]
    if history:
        for user_msg, ai_msg in history:
            messages.append({"role": "user",      "content": user_msg})
            messages.append({"role": "assistant", "content": ai_msg})
    messages.append({"role": "user", "content": message})

    # 注意：aisuite 目前不支援 streaming，這裡用模擬版
    # 真實 streaming 可改用 openai 套件的 stream=True
    response = client.chat.completions.create(
        model=f"{provider}:{model}",
        messages=messages
    )
    full_response = response.choices[0].message.content

    # 模擬逐字輸出效果
    import time
    partial = ""
    for char in full_response:
        partial += char
        yield partial
        time.sleep(0.01)  # 每個字延遲 10ms


stream_chatbot = gr.ChatInterface(
    fn=chat_stream,
    title=f"{app_title} （串流版）",
    description=app_description,
    textbox=gr.Textbox(placeholder=input_placeholder, container=False, scale=7),
    theme=gr.themes.Soft()
)

stream_chatbot.launch(share=True, debug=True)

# 🧠 核心觀念回顧

```
LLM API 的運作方式

使用者說話
    ↓
把「system + 對話歷史 + 新訊息」打包成 messages
    ↓
送給 LLM API（Groq / OpenAI / Mistral...）
    ↓
收到 AI 回應
    ↓
把新的 (user, AI) 加入對話歷史
    ↓
等待下一輪...
```

| 概念 | 說明 |
|------|------|
| System Prompt | 決定 AI 的個性、風格、限制 |
| Messages 串列 | 包含完整對話歷史，AI 「記憶」的來源 |
| AISuite | 統一界面呼叫多個 LLM 供應商 |
| Gradio ChatInterface | 自動管理對話歷史的聊天介面 |

# 🎯 本週創作任務

### 打造屬於你的 AI 角色！

修改 `system_prompt`、`app_title` 和 `app_description`，創造一個有趣的 AI 角色：

**創意方向：**
- 🎭 **角色扮演**：某個歷史人物、小說角色、動漫角色
- 🛠️ **實用工具**：英文寫作助手、程式碼解說機、選課建議機器人
- 😂 **創意搞笑**：一直把話題帶到貓身上的貓奴 AI、只會說謎語的神秘 AI
- 📝 **學習助手**：蘇格拉底式問答 AI（永遠用問句回應）

**進階挑戰：**
- 讓 App 有「切換角色」的按鈕（Gradio `Radio` 元件）
- 加入「對話歷史」的顯示和清除功能
- 讓 AI 根據對話長度，自動做對話摘要

回答：

1️⃣ 你創造了什麼角色？System prompt 怎麼寫？
2️⃣ 你用的是哪個 LLM？為什麼選它？
3️⃣ 分享最有趣的一段對話截圖！
4️⃣ AI 如何幫助你完成？